# Prompt Engineering Fundamentals

Master the techniques that turn a mediocre LLM response into a production-quality one.

**Topics:** Zero-shot, few-shot, chain-of-thought, structured output, role prompting, JSON mode, prompt templates

## 1. Zero-Shot vs Few-Shot Prompting

In [ ]:
# Zero-shot: just describe the task
ZERO_SHOT = """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: {review}
Sentiment:"""

# Few-shot: show examples before the actual input
FEW_SHOT = """Classify the sentiment. Output only the label.

Review: The battery lasts all day and the camera is stunning.
Sentiment: POSITIVE

Review: Completely stopped working after 2 weeks. Total waste of money.
Sentiment: NEGATIVE

Review: It arrived on time and the packaging was fine.
Sentiment: NEUTRAL

Review: {review}
Sentiment:"""

test_reviews = [
    'Best purchase I have made all year!',
    'The screen cracked on day one.',
    'Standard product, does what it says.',
]

print('Prompt comparison:')
print(f'  Zero-shot tokens: ~{len(ZERO_SHOT.split())}')
print(f'  Few-shot tokens:  ~{len(FEW_SHOT.split())}')
print()
print('When to use each:')
print('  Zero-shot: simple, clear tasks; reduces tokens/cost')
print('  Few-shot: complex formatting, domain-specific labels, edge cases')
print()
print('Rule of thumb: start zero-shot, add examples only when accuracy is insufficient.')

## 2. Chain-of-Thought Reasoning

In [ ]:
# Direct answer (often wrong on complex problems)
DIRECT = """A store has 15 apples. They sell 8 and receive a shipment of 20 more.
Then they sell half of what they have. How many apples remain?
Answer:"""

# Chain-of-thought: ask the model to think step by step
COT = """A store has 15 apples. They sell 8 and receive a shipment of 20 more.
Then they sell half of what they have. How many apples remain?

Let's think step by step:"""

# Zero-shot CoT — just add the magic phrase
COT_ZERO_SHOT = """Q: {question}

A: Let's think step by step."""

# Self-consistency CoT: generate N solutions, take majority vote
def self_consistency_prompt(question: str, n_paths: int = 5) -> str:
    return f"""Solve the following problem. Show your reasoning.

Problem: {question}

Reasoning:"""

# CoT improves accuracy by:
improvements = {
    'Math word problems': '+30-40% accuracy',
    'Multi-step reasoning': '+25-35% accuracy',
    'Code debugging': '+20-30% accuracy',
    'Simple classification': '~0% (not needed)',
}

print('Chain-of-Thought accuracy improvements:')
for task, gain in improvements.items():
    print(f'  {task:<30}: {gain}')
print()
print('Magic phrases that trigger CoT:')
for phrase in ['Let\'s think step by step.', 'Think carefully.', 'Reason through this.']:
    print(f'  "{phrase}"')

## 3. Structured Output with JSON Mode

In [ ]:
import json
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

# Method 1: JSON mode (OpenAI gpt-4o)
EXTRACTION_PROMPT = """Extract the following fields from the job posting.
Return valid JSON with keys: title, company, location, salary_range, required_skills (list), experience_years (int).

Job posting:
{posting}"""

def extract_job_info(posting: str) -> dict:
    """Extract structured data from free text using JSON mode."""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You extract structured information. Always output valid JSON.'},
            {'role': 'user', 'content': EXTRACTION_PROMPT.format(posting=posting)},
        ],
        response_format={'type': 'json_object'},  # <-- enforces JSON output
    )
    return json.loads(response.choices[0].message.content)

# Sample posting
sample_posting = """
Senior ML Engineer at TechCorp (San Francisco, CA)
Salary: $180,000 - $220,000
We need 5+ years experience with Python, PyTorch, and Kubernetes.
Experience with MLOps (MLflow, DVC) is a big plus.
"""

expected_output = {
    'title': 'Senior ML Engineer',
    'company': 'TechCorp',
    'location': 'San Francisco, CA',
    'salary_range': '$180,000 - $220,000',
    'required_skills': ['Python', 'PyTorch', 'Kubernetes', 'MLflow', 'DVC'],
    'experience_years': 5,
}

print('Input posting:')
print(sample_posting.strip())
print()
print('Expected structured output:')
print(json.dumps(expected_output, indent=2))
print()
print('Key: response_format={"type": "json_object"} guarantees parseable JSON.')

## 4. Role and Persona Prompting

In [ ]:
# Role prompting dramatically changes response quality for specialized tasks

PERSONAS = {
    'code_reviewer': """You are a senior software engineer with 15 years of experience.
You review code for: correctness, security vulnerabilities, performance, and maintainability.
Be direct and specific. Format issues as a numbered list. Always suggest fixes.""",

    'data_scientist': """You are a senior data scientist specializing in ML systems.
You explain complex concepts clearly using analogies and concrete examples.
When answering questions, always mention: assumptions, limitations, and when NOT to use a technique.""",

    'socratic_tutor': """You are a Socratic tutor. Never give direct answers.
Instead, ask guiding questions that lead the student to discover the answer themselves.
Keep questions focused and build on the student's responses.""",

    'devil_advocate': """You are a devil's advocate. Your job is to find weaknesses in proposals.
Challenge assumptions, identify edge cases, and raise potential failure modes.
Be constructive — the goal is to strengthen ideas, not destroy them.""",
}

# Meta-prompt for dynamic persona generation
def build_expert_system_prompt(domain: str, task: str) -> str:
    return f"""You are a world-class expert in {domain} with deep practical experience.
Your goal: {task}
Communication style: precise, actionable, example-driven.
Always acknowledge uncertainty when present."""

print('Available personas:')
for name, prompt in PERSONAS.items():
    print(f'  {name}: {prompt[:60].strip()}...')
print()
print('Dynamic persona example:')
print(build_expert_system_prompt('RAG systems', 'debug why retrieval quality is poor'))

## 5. Prompt Templates with Jinja2

In [ ]:
from jinja2 import Template, Environment, FileSystemLoader
from dataclasses import dataclass
from typing import list

@dataclass
class PromptTemplate:
    """Versioned, testable prompt template."""
    name: str
    version: str
    template_str: str
    
    def render(self, **kwargs) -> str:
        return Template(self.template_str).render(**kwargs)
    
    def __repr__(self) -> str:
        return f'PromptTemplate({self.name} v{self.version})'

# Template with conditionals and loops
RAG_TEMPLATE = PromptTemplate(
    name='rag_qa',
    version='2.1',
    template_str="""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have enough information."

{% if persona %}You are acting as: {{ persona }}\n{% endif %}

Context:
{% for i, doc in documents %}
[{{ i+1 }}] {{ doc }}
{% endfor %}

Question: {{ question }}

{% if require_citation %}Cite your sources using [1], [2] etc.{% endif %}

Answer:""",
)

rendered = RAG_TEMPLATE.render(
    persona='ML Engineer',
    documents=[
        ('Transformers use self-attention to process sequences in parallel.',),
        ('BERT is pre-trained with masked language modeling and next sentence prediction.',),
    ],
    question='What makes transformers different from RNNs?',
    require_citation=True,
)

print(f'Template: {RAG_TEMPLATE}')
print(f'Rendered ({len(rendered)} chars):')
print('-' * 40)
print(rendered[:400] + '...')

## Exercises

1. **Prompt comparison:** Write a benchmark function that sends the same classification task with zero-shot vs 3-shot vs 5-shot prompts, measures accuracy on 10 test examples, and plots the results.

2. **Auto-CoT:** Implement a function that automatically generates few-shot chain-of-thought examples by asking the LLM to solve training examples and show its reasoning, then uses those examples for test inference.

3. **Prompt registry:** Design a `PromptRegistry` class that stores versioned prompts in a YAML file, loads them by name/version, and logs which version was used for each API call (for reproducibility).